# 06 — Text normalization (Phase 6)

## Concept
**Normalization** = rewriting text into a standard form so that the *same* thing written *differently*
becomes *identical*: `Pvt.` / `PVT` / `Private` → `private`.

## Why do we need it?
Our Jaccard similarity only counts **identical** words. Every variant spelling of the same word
costs us similarity (missed matches) and every lost rare word costs us blocking recall.

## The danger
Normalizing too aggressively **destroys information** — e.g. removing legal suffixes makes
`prairie anchor home care llc` and `prairie anchor home care ltd` identical, but Phase 5 showed they are
*different* businesses. So **every step is an experiment**: we add it alone to the baseline and keep it only if the
validation score goes up.

## How the experiments run
All steps live in `src/preprocessing.py`. The Phase 5 pipeline is packaged as `src/rule_experiment.run(name_steps, address_steps)`,
which rebuilds blocking and scoring with the chosen steps, tunes the threshold on the 20k **tuning** entities and scores the
20k **small-validation** entities. The script `experiments/phase6_normalization.py` runs one experiment per step
(~4 minutes each) and saves the results to `experiments/phase6_results.tsv`.

Speed-up used: the pool is cached as **Parquet** (`cache/train_pool.parquet`) — a compressed column format that loads in ~2 s
instead of ~40 s for the TSV files.

In [1]:
import inspect
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import pyarrow.parquet as pq

from src import preprocessing
from src.preprocessing import normalize

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)

# 1 million Source 2 records + 1 million Source 3 records are plenty for examples
pool_file = pq.ParquetFile(PROJECT_ROOT / "cache" / "train_pool.parquet")
examples_pool = pd.concat([pool_file.read_row_group(0).to_pandas(), pool_file.read_row_group(6).to_pandas()], ignore_index=True)
print(len(examples_pool), "pool records loaded for examples")

2000000 pool records loaded for examples


### The normalization pipeline

`normalize(text, steps)` always lowercases and turns symbols into spaces. The optional steps run in a fixed order:
first the ones that need the raw text (dots, `&`, `.com`, accents …), then the ones that work on clean words.

In [2]:
print(inspect.getsource(preprocessing.normalize))

def normalize(text, steps=()):
    """Lowercase + optional raw steps + symbol removal + optional word steps."""
    unknown = set(steps) - set(STEP_ORDER)
    if unknown:
        raise ValueError(f"unknown normalization steps: {unknown}")
    text = text.str.lower()
    for step in RAW_STEPS:
        if step in steps:
            text = RAW_STEPS[step](text)
    text = basic_clean(text)
    for step in WORD_STEPS:
        if step in steps:
            text = WORD_STEPS[step](text)
    return text



In [3]:
def show_step(step, column, pattern, n=6, case=False):
    rows = examples_pool[examples_pool[column].str.contains(pattern, case=case, regex=True)]
    rows = rows.sample(min(n, len(rows)), random_state=0)
    return pd.DataFrame({
        "raw": rows[column],
        "baseline cleaning": normalize(rows[column]),
        f"+ {step}": normalize(rows[column], [step]),
    }).reset_index(drop=True)

---
## The candidate steps, with real before / after examples

### `trade_name`

Source 3 prefixes an invented brand to the real name: `X dba Y`, `X formerly known as Y`, `X née Y`. We keep only the part **after** the marker (only if something comes before it, so `dba brothers pvt ltd` is untouched).

In [4]:
show_step("trade_name", "business_name", r"\sdba\s|formerly|\snée\s|trading as|\st/a\s|\saka\s")

,raw,baseline cleaning,+ trade_name
0,Rizaquox aka Worlds Developers Private Limited,rizaquox aka worlds developers private limited,worlds developers private limited
1,Zetajax aka Regional Tri-State Elm PLLC,zetajax aka regional tri state elm pllc,regional tri state elm pllc
2,Wexgild trading as Chiropractic Anchor Clinic,wexgild trading as chiropractic anchor clinic,chiropractic anchor clinic
3,Korvantage DBA Sanand India Private Limited,korvantage dba sanand india private limited,sanand india private limited
4,Aviquo formerly known as Beverlee's Tax LLC,aviquo formerly known as beverlee s tax llc,beverlee s tax llc
5,Ariajax dba Lower Rocky Shore,ariajax dba lower rocky shore,lower rocky shore


### `invisible`

Zero-width characters inside Indian-script words. Baseline cleaning turned them into a *space*, splitting one word into two; we delete them instead.

In [5]:
show_step("invisible", "business_name", r"‌|‍")

,raw,baseline cleaning,+ invisible
0,జైన్ సాఫ్ట్‌వేర్ ప్రైవేట్ లిమిటెడ్,జైన్ సాఫ్ట్ వేర్ ప్రైవేట్ లిమిటెడ్,జైన్ సాఫ్ట్వేర్ ప్రైవేట్ లిమిటెడ్
1,శక్తి ఇన్వెస్ట్‌మెంట్ ఎల్‌ఎల్‌పీ,శక్తి ఇన్వెస్ట్ మెంట్ ఎల్ ఎల్ పీ,శక్తి ఇన్వెస్ట్మెంట్ ఎల్ఎల్పీ
2,ಶಿವಂ ಕನ್‌ಸ್ಟ್ರಕ್ಷನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,ಶಿವಂ ಕನ್ ಸ್ಟ್ರಕ್ಷನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,ಶಿವಂ ಕನ್ಸ್ಟ್ರಕ್ಷನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್
3,ಇಂಟರ್‌ನ್ಯಾಷನಲ್ ಡೆವಲಪರ್ಸ್ ಎಲ್ಎಲ್‌ಪಿ,ಇಂಟರ್ ನ್ಯಾಷನಲ್ ಡೆವಲಪರ್ಸ್ ಎಲ್ಎಲ್ ಪಿ,ಇಂಟರ್ನ್ಯಾಷನಲ್ ಡೆವಲಪರ್ಸ್ ಎಲ್ಎಲ್ಪಿ
4,Laxmi ಕನ್‌ಸ್ಟ್ರಕ್ಷನ್ Private Limited,laxmi ಕನ್ ಸ್ಟ್ರಕ್ಷನ್ private limited,laxmi ಕನ್ಸ್ಟ್ರಕ್ಷನ್ private limited
5,స్కై ఇన్వెస్ట్‌మెంట్ ప్రైవేట్ లిమిటెడ్,స్కై ఇన్వెస్ట్ మెంట్ ప్రైవేట్ లిమిటెడ్,స్కై ఇన్వెస్ట్మెంట్ ప్రైవేట్ లిమిటెడ్


### `accents`

Fake accents (`límited`, `ínc`) and real French accents (`école`). `é → e`. Only Latin accents are removed; Indian vowel signs stay.

In [6]:
show_step("accents", "business_name", r"[À-ɏ]", case=True)

,raw,baseline cleaning,+ accents
0,Gonzalez and Táylor LLC,gonzalez and táylor llc,gonzalez and taylor llc
1,"Nelson, Williams & Báylor LLC",nelson williams báylor llc,nelson williams baylor llc
2,Padilla & Burdette Ínc,padilla burdette ínc,padilla burdette inc
3,Express Áatitmn Outdoor LLC,express áatitmn outdoor llc,express aatitmn outdoor llc
4,Shuddhi Índia Pvt Ltd,shuddhi índia pvt ltd,shuddhi india pvt ltd
5,*** Behavioral Group Pártners,behavioral group pártners,behavioral group partners


### `ampersand`

`&` becomes the word `and`, so `smith & sons` matches `smith and sons`.

In [7]:
show_step("ampersand", "business_name", r"&")

,raw,baseline cleaning,+ ampersand
0,Dr Brook & Partners Exports Private,dr brook partners exports private,dr brook and partners exports private
1,"LLC Kaufman, Phillips & Gfual Highland",llc kaufman phillips gfual highland,llc kaufman phillips and gfual highland
2,Shabazz-& Csto,shabazz csto,shabazz and csto
3,Sri Cg & Co,sri cg co,sri cg and co
4,Swarajya & Co Overseas Ltd,swarajya co overseas ltd,swarajya and co overseas ltd
5,Cloud & Partners Private,cloud partners private,cloud and partners private


### `web`

Website names: drop `www.` and the `.com` / `.in` / `.net` … ending, so `rcmedical.com` becomes the single word `rcmedical`.

In [8]:
show_step("web", "business_name", r"\.com\b|www\.")

,raw,baseline cleaning,+ web
0,directaltenergymodern.com,directaltenergymodern com,directaltenergymodern
1,platinumlearning.com,platinumlearning com,platinumlearning
2,leacockkpet.com,leacockkpet com,leacockkpet
3,primarycareclinic.com,primarycareclinic com,primarycareclinic
4,Urology Partners LLC | www.urologypa.com,urology partners llc www urologypa com,urology partners llc urologypa
5,LITTLEICECREAM.COM,littleicecream com,littleicecream


### `dots`

Delete dots instead of turning them into spaces: `L.L.C.` → `llc` (not `l l c`), `Pvt.` → `pvt`.

In [9]:
show_step("dots", "business_name", r"\b[a-z]\.[a-z]\.")

,raw,baseline cleaning,+ dots
0,"Celestina Mccreery, M.D., P.C. Center",celestina mccreery m d p c center,celestina mccreery md pc center
1,Sri Lotus Global Jewellers L.L.P.,sri lotus global jewellers l l p,sri lotus global jewellers llp
2,Q/C-P.C. Center,q c p c center,q c pc center
3,Divine & 5ons L.L.P.,divine 5ons l l p,divine 5ons llp
4,Clayton Quality TAX Service L.L.C.,clayton quality tax service l l c,clayton quality tax service llc
5,"GUSSI LANCE, P.A. COASTAL P.L.L.C.",gussi lance p a coastal p l l c,gussi lance pa coastal pllc


### `legal_canonical`

One spelling per legal form: `pvt → private`, `ltd → limited`, `inc → incorporated`, `corp → corporation`, `co → company`.

In [10]:
show_step("legal_canonical", "business_name", r"\bpvt\b|\bltd\b|\binc\b|\bcorp\b")

,raw,baseline cleaning,+ legal_canonical
0,(india) Ássociation Digitech Pvt. Ltd.,india ássociation digitech pvt ltd,india ássociation digitech private limited
1,IR Property Solutions Associates (Corp),ir property solutions associates corp,ir property solutions associates corporation
2,Quality Wealth Collective Ltd,quality wealth collective ltd,quality wealth collective limited
3,Great Oil Inc.,great oil inc,great oil incorporated
4,Gurgaon Foundation Ltd,gurgaon foundation ltd,gurgaon foundation limited
5,Select Musltang Inc.,select musltang inc,select musltang incorporated


### `legal_remove`

The opposite idea: **delete** legal words so only the distinctive part of the name is compared.

In [11]:
show_step("legal_remove", "business_name", r"\bpvt\b|\bllc\b|\binc\b|\blimited\b")

,raw,baseline cleaning,+ legal_remove
0,Smt Lotus Infrastructure Private (Limited),smt lotus infrastructure private limited,smt lotus infrastructure
1,Dhanvantari Lifestyle Private Limited,dhanvantari lifestyle private limited,dhanvantari lifestyle
2,O T & D Enterprise Llc,o t d enterprise llc,o t d enterprise
3,Eagle Vincom Pvt.,eagle vincom pvt,eagle vincom
4,Limited Constructions Private Service,limited constructions private service,constructions service
5,Orthopedic Partners Downtown Inc.,orthopedic partners downtown inc,orthopedic partners downtown


### `address_abbrev`

`st → street`, `rd → road`, `dr → drive`, `ave → avenue`, `ln → lane`, `blvd → boulevard`, French `r → rue`, `bd → boulevard`, `av → avenue`. **Only in addresses** (in names `dr` means *Doctor*).

In [12]:
show_step("address_abbrev", "business_address", r"\bst\b|\brd\b|\bave\b|\bdr\b")

,raw,baseline cleaning,+ address_abbrev
0,"1329 HIGHLAND AVE, CLARKSTON, WA",1329 highland ave clarkston wa,1329 highland avenue clarkston wa
1,"119 1/2 Sugar Cane Dr, Youngstown, Ohio",119 1 2 sugar cane dr youngstown ohio,119 1 2 sugar cane drive youngstown ohio
2,"Wisconsin, City Of Oshkosh, 1610 Jackson St",wisconsin city of oshkosh 1610 jackson st,wisconsin city of oshkosh 1610 jackson street
3,"York Ave, Salem, Oregon",york ave salem oregon,york avenue salem oregon
4,"405- High St Street, Lordsburg, New Mexico",405 high st street lordsburg new mexico,405 high street street lordsburg new mexico
5,"##471 Green Rd, null, Rutherfordton, North Carolina",471 green rd null rutherfordton north carolina,471 green road null rutherfordton north carolina


### `null`

Delete the placeholder word `null` / `<NULL>` inside addresses.

In [13]:
show_step("null", "business_address", r"\bnull\b")

,raw,baseline cleaning,+ null
0,"13 Knights Way, <NULL>, Bethlehem, New York",13 knights way null bethlehem new york,13 knights way bethlehem new york
1,"5633 Muirfield Dr Southwest, <NULL>, Cedar Rapids, Iowa",5633 muirfield dr southwest null cedar rapids iowa,5633 muirfield dr southwest cedar rapids iowa
2,"331 HIGHWAY 62, <NULL>, YANCEYVILLE CITY, NC",331 highway 62 null yanceyville city nc,331 highway 62 yanceyville city nc
3,"null, 5/104, BESIDE SBI BANK, SRIKAKULAM ROAD, RAJAM, Andhra Pradesh",null 5 104 beside sbi bank srikakulam road rajam andhra pradesh,5 104 beside sbi bank srikakulam road rajam andhra pradesh
4,"186 BOURDON STREET, <NULL>, SHOW LOW, AZ",186 bourdon street null show low az,186 bourdon street show low az
5,"C4-91, YAMUNA VIHAR, SHAHDARA NEAR RATH WALA MANDIR, NORTH EAST DELHI, NULL,...",c4 91 yamuna vihar shahdara near rath wala mandir north east delhi null दिल्ली,c4 91 yamuna vihar shahdara near rath wala mandir north east delhi दिल्ली


### `zeros`

Remove leading zeros from numbers: `N°022` → `22`, `038` → `38`.

In [14]:
show_step("zeros", "business_address", r"\b0\d")

,raw,baseline cleaning,+ zeros
0,"00502, A-WING, 5TH FLOOR, RAMKRISHNA CHSL, BABHAI NAKA, L T ROAD, BORIVALI W...",00502 a wing 5th floor ramkrishna chsl babhai naka l t road borivali west mu...,502 a wing 5th floor ramkrishna chsl babhai naka l t road borivali west mumb...
1,"Shop-514, Shah Prima, Plot No.-13 Sector-02, Khaghar, Raigarh, महाराष्ट्र",shop 514 shah prima plot no 13 sector 02 khaghar raigarh महाराष्ट्र,shop 514 shah prima plot no 13 sector 2 khaghar raigarh महाराष्ट्र
2,"#02-06-091/1-1B (2), Kurmawada, Jangoan, Telangana., Jangoan, Warangal Urban...",02 06 091 1 1b 2 kurmawada jangoan telangana jangoan warangal urban tg,2 6 91 1 1b 2 kurmawada jangoan telangana jangoan warangal urban tg
3,"Block G-00529 A-25 Yeshwant Palaceshree Rang Chsl Opp Shreerang Bus Stop, Gr...",block g 00529 a 25 yeshwant palaceshree rang chsl opp shreerang bus stop gre...,block g 529 a 25 yeshwant palaceshree rang chsl opp shreerang bus stop great...
4,"0013919 Michaux View Way, Chesterfield County, Virginia",0013919 michaux view way chesterfield county virginia,13919 michaux view way chesterfield county virginia
5,"Hyderabad, TG, Flat No.09",hyderabad tg flat no 09,hyderabad tg flat no 9


---
## Results: each step alone vs the baseline

Every row is a full run: blocking + Jaccard scoring + threshold tuned on the tuning set, scored on the small validation set.
`Δ` = change vs the baseline.

In [15]:
results = pd.read_csv(PROJECT_ROOT / "experiments" / "phase6_results.tsv", sep="\t", keep_default_na=False)
single = results[~results["experiment"].str.startswith("combo")].set_index("experiment")
base = single.loc["baseline"]
single["Δ F0.5"] = (single["validation_F0.5"] - base["validation_F0.5"]).round(4)
single["Δ candidate recall"] = (single["validation_candidate_recall"] - base["validation_candidate_recall"]).round(4)
single[["validation_F0.5", "Δ F0.5", "validation_candidate_recall", "Δ candidate recall",
        "validation_precision", "validation_recall", "threshold"]].sort_values("Δ F0.5", ascending=False)

,validation_F0.5,Δ F0.5,validation_candidate_recall,Δ candidate recall,validation_precision,validation_recall,threshold
experiment,,,,,,,
address_abbrev,0.5307,0.0029,0.5867,-0.0011,0.8245,0.4539,0.55
accents,0.5282,0.0004,0.5875,-0.0003,0.8343,0.4410,0.55
invisible,0.5278,0.0000,0.5878,0.0000,0.7381,0.5059,0.45
baseline,0.5278,0.0000,0.5878,0.0000,0.7381,0.5059,0.45
web,0.5278,0.0000,0.5878,0.0000,0.7376,0.5065,0.45
null,0.5277,-0.0001,0.5879,0.0001,0.7380,0.5063,0.45
ampersand,0.5276,-0.0002,0.5878,0.0000,0.7373,0.5069,0.45
trade_name,0.5274,-0.0004,0.5874,-0.0004,0.7365,0.5057,0.45
zeros,0.5267,-0.0011,0.5887,0.0009,0.7376,0.5083,0.45


**Reading the single-step results**
- Only **`address_abbrev`** clearly helps on its own (+0.003): `st`/`street`, `rd`/`road` now match, so true pairs get higher address similarity.
- Most other steps change almost nothing (±0.001). Why? Indian-script words and glued website names could never share a word with the
  English S1 name anyway, so cleaning them does not change Jaccard. And 41% of true matches never even reach the scorer (blocking recall ~59%).
- Steps that make **legal words match more often hurt**: `legal_canonical` (−0.004), `dots` (−0.003, turns `l l c` into `llc`),
  and especially `legal_remove` (−0.011). Legal words are shared by thousands of unrelated businesses — making them match more
  raises the similarity of *wrong* candidates (more false merges), and removing them throws away the `llc` vs `ltd` difference
  that separates look-alike businesses.
- Several steps move the best threshold from 0.45 to 0.55, trading recall for precision.

**Decision rule for combinations:** combine the steps that *helped* (`address_abbrev`, `accents`) and, separately, add the *neutral* ones that are
logically sound and will matter later for character-level features (`trade_name`, `invisible`, `web`, `null`, `zeros`). Never use the harmful ones.

In [16]:
combined = results[results["experiment"].str.startswith("combo")].set_index("experiment")
combined["Δ F0.5"] = (combined["validation_F0.5"] - base["validation_F0.5"]).round(4)
combined[["name_steps", "address_steps", "validation_F0.5", "Δ F0.5", "validation_candidate_recall",
          "validation_precision", "validation_recall", "threshold"]]

,name_steps,address_steps,validation_F0.5,Δ F0.5,validation_candidate_recall,validation_precision,validation_recall,threshold
experiment,,,,,,,,
combo_0_baseline,,,0.5278,0.0000,0.5878,0.7381,0.5059,0.45
combo_1_helped,accents,accents+address_abbrev,0.5362,0.0084,0.5895,0.8244,0.4661,0.55
combo_2_helped_plus_neutral,trade_name+invisible+accents+web,invisible+accents+address_abbrev+null+zeros,0.5378,0.0100,0.5895,0.8235,0.4699,0.55


---
## Is the improvement real, or luck? — a paired bootstrap

### Concept
Our validation set is a *sample* of 20,000 entities. A different sample would give slightly different scores,
so a difference of +0.003 might just be luck.

**Paired** comparison: both versions are scored on the **same** 20,000 entities, so we look at the per-entity
*difference* `F0.5(new) − F0.5(baseline)`. Most entities have difference 0 (both versions answer the same).

**Bootstrap**: re-draw 20,000 entities *with replacement* 1,000 times and recompute the average difference each time.
The middle 95% of those 1,000 averages is a **95% confidence interval**.
If the whole interval is above 0, the improvement is very unlikely to be luck.

In [17]:
import numpy as np

per_entity_dir = PROJECT_ROOT / "output" / "experiments" / "phase6"
baseline_scores = pd.read_csv(per_entity_dir / "combo_0_baseline.tsv", sep="\t")

rows = []
for name in ["combo_1_helped", "combo_2_helped_plus_neutral"]:
    other = pd.read_csv(per_entity_dir / f"{name}.tsv", sep="\t")
    both = baseline_scores.merge(other, on="source1_entity_id", suffixes=("_base", "_new"))
    diff = (both["f05_new"] - both["f05_base"]).to_numpy()

    rng = np.random.default_rng(0)
    boot_means = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(1000)]
    low, high = np.percentile(boot_means, [2.5, 97.5])
    rows.append({
        "experiment": name,
        "mean difference": round(diff.mean(), 4),
        "95% CI low": round(low, 4),
        "95% CI high": round(high, 4),
        "entities better": int((diff > 0).sum()),
        "entities worse": int((diff < 0).sum()),
        "entities unchanged": int((diff == 0).sum()),
    })
pd.DataFrame(rows).set_index("experiment")

,mean difference,95% CI low,95% CI high,entities better,entities worse,entities unchanged
experiment,,,,,,
combo_1_helped,0.0084,0.0050,0.0116,3926,3265,12809
combo_2_helped_plus_neutral,0.0100,0.0065,0.0134,4040,3256,12704


Both combinations beat the baseline with a 95% confidence interval **entirely above 0**, so the gain is not luck.
Combination 2 is best on the **tuning** set too (0.5377 vs 0.5357), so choosing it does not rely on the validation labels.

---
# Summary — Phase 6

| | Validation F0.5 | Candidate recall |
|---|---|---|
| Baseline (Phase 5) | 0.5278 | 0.588 |
| + accents + address abbreviations | 0.5362 | 0.590 |
| **+ trade names, invisible chars, web names, `null`, leading zeros** | **0.5378** | **0.590** |

**Chosen normalization** (saved as `NAME_STEPS` / `ADDRESS_STEPS` in `src/preprocessing.py`):
- names: `trade_name`, `invisible`, `accents`, `web`
- addresses: `invisible`, `accents`, `address_abbrev`, `null`, `zeros`

**Rejected:** `legal_canonical`, `legal_remove`, `dots`, `ampersand` — legal words must stay as they are.

**Lessons**
1. Measure every cleaning step; "obviously good" steps can hurt (legal words).
2. Small differences need a significance check (paired bootstrap).
3. Normalization gives only +0.010 here because **blocking recall (~59%) is the real bottleneck** → Phase 7.